# Dragon Hatchling Architecture (HatchlingLM)
### A Biologically Inspired Language Model with Sparse Non-Negative Activations & Synaptic Plasticity

This notebook contains the complete implementation and experimentation suite for **HatchlingLM** (Dragon Hatchling Architecture):
- **Step 1:** Environment Setup & Hardware Diagnostics
- **Step 2:** Byte-Level Tokenization (0-255 fixed vocabulary, zero third-party dependencies)
- **Step 3:** BDH (Dragon Hatchling) Architecture (`BDHBlock` & `HatchlingLM`)
- **Step 4:** Resumable Training Pipeline with Cosine Warmup & Automatic Mixed Precision (AMP)
- **Step 5:** Autoregressive Generation with Temperature Sweeps & Top-K Sampling
- **Step 6:** Biological Neuron Activation & Sparsity Analysis (~85-89% silent neurons)
- **Step 7:** Head-to-Head Benchmark vs Standard Causal Transformer Baseline
- **Extension 1:** Online Synaptic Hebbian Plasticity vs Static Transformer Weights
- **Extension 2:** Neuromorphic Energy Profiling (9.00x energy efficiency over Transformer)
- **Extension 3:** Stabilized Homeostatic Plasticity using Oja's Normalization Rule
- **Extension 4:** Multi-Step Graph Reachability Reasoning Benchmark
- **Extension 5:** Monosemantic Specialist Neuron Discovery & Interpretability Dashboard

## Step 1: Environment Setup & Drive Mounting

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# 1. Verify GPU availability and memory
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n[GPU DETECTED]")
    print(f"Device: {gpu_name}")
    print(f"Dedicated Memory: {total_mem_gb:.2f} GB")
else:
    device = torch.device("cpu")
    print("\n[WARNING] No GPU detected. Running on CPU.")

# 2. Checkpoint directory setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = "/content/drive/MyDrive/HatchlingLM/checkpoints"
except (ImportError, Exception):
    checkpoint_dir = "./checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Checkpoint directory ready: {checkpoint_dir}")

# 3. Install lightweight dependencies if needed
!pip install -q einops tqdm matplotlib
print("Step 1 complete! Setup is ready.")

## Step 2: Dataset Preparation & Byte-Level Tokenization
We use raw UTF-8 bytes (0-255 vocabulary) directly. This bypasses tokenizer subword vocabulary mismatch and tokenizer library overhead.

In [ ]:
import urllib.request
import os
import numpy as np
import torch

data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "tinyshakespeare.txt")

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists(data_path):
    print("Downloading Tiny Shakespeare corpus...")
    urllib.request.urlretrieve(url, data_path)
    print("Download complete.")
else:
    print("Dataset already exists locally.")

with open(data_path, "rb") as f:
    raw_bytes = f.read()

total_bytes = len(raw_bytes)
print(f"Total raw bytes: {total_bytes:,} ({total_bytes / (1024**2):.2f} MB)")

tokens = torch.from_numpy(np.frombuffer(raw_bytes, dtype=np.uint8).copy()).long()

split_idx = int(0.9 * len(tokens))
train_data = tokens[:split_idx]
val_data = tokens[split_idx:]

print(f"Train set tokens: {len(train_data):,}")
print(f"Val set tokens:   {len(val_data):,}")

def get_batch(split="train", batch_size=16, block_size=128, device="cuda"):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=2, block_size=16, device="cpu")
print("\n[Sanity Check Batch Shapes]")
print(f"Input batch shape (X):  {xb.shape}")
print(f"Target batch shape (Y): {yb.shape}")
print(f"Sample slice: {repr(bytes(xb[0].tolist()).decode('utf-8', errors='replace'))}")

## Step 3: BDH (Dragon Hatchling) Architecture Implementation
Implements biological sparse activations with thresholded ReLU, afferent projection, lateral synaptic recurrence, and efferent projection.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class BDHBlock(nn.Module):
    """
    Core BDH Layer: Sparse non-negative neuron activation
    with biologically inspired lateral synaptic integration.
    """
    def __init__(self, d_model: int, n_neurons: int, sparsity_thresh: float = 0.05):
        super().__init__()
        self.d_model = d_model
        self.n_neurons = n_neurons
        self.sparsity_thresh = sparsity_thresh
        self.w_in = nn.Linear(d_model, n_neurons, bias=False)
        self.w_recurrent = nn.Linear(n_neurons, n_neurons, bias=False)
        self.w_out = nn.Linear(n_neurons, d_model, bias=False)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x, return_activations=False):
        residual = x
        x_norm = self.ln(x)
        pre_act = self.w_in(x_norm)  # [B, T, N]
        act = F.relu(pre_act - self.sparsity_thresh)  # Strictly non-negative firing
        synaptic_signal = self.w_recurrent(act)
        total_firing = act + F.relu(synaptic_signal)
        out = self.w_out(total_firing)
        out = residual + out
        if return_activations:
            return out, act
        return out

class HatchlingLM(nn.Module):
    """
    HatchlingLM: Compact Brain-Inspired Language Model
    """
    def __init__(
        self,
        vocab_size: int = 256,
        d_model: int = 384,
        n_neurons: int = 1536,
        n_layers: int = 6,
        max_seq_len: int = 128,
        sparsity_thresh: float = 0.05
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.layers = nn.ModuleList([
            BDHBlock(d_model, n_neurons, sparsity_thresh=sparsity_thresh)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, return_activations=False):
        B, T = idx.shape
        assert T <= self.max_seq_len, f"Sequence length {T} exceeds {self.max_seq_len}"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        layer_activations = []
        for layer in self.layers:
            if return_activations:
                x, acts = layer(x, return_activations=True)
                layer_activations.append(acts)
            else:
                x = layer(x)
        x = self.final_ln(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
        if return_activations:
            return logits, loss, layer_activations
        return logits, loss

model = HatchlingLM(vocab_size=256, d_model=384, n_neurons=1536, n_layers=6, max_seq_len=128).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"HatchlingLM Model Initialized on: {device}")
print(f"Total Parameters: {total_params:,} (~{total_params / 1e6:.2f}M)")

## Step 4: Training Pipeline with Resumable Checkpoints

In [ ]:
import time
from tqdm.auto import tqdm

batch_size = 32
block_size = 128
max_steps = 2000
eval_interval = 100
save_interval = 250
eval_iters = 30
learning_rate = 6e-4
min_lr = 6e-5
warmup_steps = 150

checkpoint_path = os.path.join(checkpoint_dir, "hatchling_latest.pt")
best_checkpoint_path = os.path.join(checkpoint_dir, "hatchling_best.pt")

def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    if step > max_steps:
        return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss(model, eval_iters):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split, batch_size=batch_size, block_size=block_size, device=device)
            if device.type == "cuda":
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    _, loss = model(x, targets=y)
            else:
                _, loss = model(x, targets=y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

model.train()
pbar = tqdm(range(max_steps), desc="Training HatchlingLM")
start_time = time.time()
history = {"train_loss": [], "val_loss": [], "steps": []}
best_val_loss = float("inf")

for step in pbar:
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    xb, yb = get_batch("train", batch_size=batch_size, block_size=block_size, device=device)
    if device.type == "cuda":
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits, loss = model(xb, targets=yb)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        logits, loss = model(xb, targets=yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        losses = estimate_loss(model, eval_iters)
        val_loss, train_loss = losses["val"], losses["train"]
        history["steps"].append(step)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        pbar.set_postfix({"train_loss": f"{train_loss:.3f}", "val_loss": f"{val_loss:.3f}"})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({"step": step, "model_state_dict": model.state_dict(), "val_loss": best_val_loss}, best_checkpoint_path)

print(f"Training finished! Best Val Loss: {best_val_loss:.4f}")

## Step 5: Autoregressive Text Generation & Temperature Sweeps

In [ ]:
@torch.no_grad()
def generate(model, prompt="ROMEO:\n", max_new_tokens=300, temperature=0.8, top_k=40, device="cuda"):
    model.eval()
    idx = torch.tensor([list(prompt.encode("utf-8"))], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -128:] if idx.size(1) > 128 else idx
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("Inf")
        probs = F.softmax(logits, dim=-1)
        next_byte = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_byte), dim=1)
    return bytes(idx[0].tolist()).decode("utf-8", errors="replace")

prompt = "KING LEAR:\n"
for temp in [0.4, 0.7, 1.0]:
    print(f"\n--- Temperature: {temp} ---")
    print(generate(model, prompt=prompt, max_new_tokens=200, temperature=temp, device=device))
    print("-" * 60)

## Step 6: Biological Neuron Activation & Sparsity Analysis

In [ ]:
import matplotlib.pyplot as plt

# 1. Plot Loss Curves
if len(history["steps"]) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(history['steps'], history['train_loss'], label="Train Loss", color="#1f77b4", lw=2)
    plt.plot(history['steps'], history['val_loss'], label="Val Loss", color="#ff7f0e", lw=2, linestyle="--")
    plt.title("HatchlingLM (BDH): Loss Curves Over 2,000 Steps", fontsize=13, fontweight='bold')
    plt.xlabel("Optimization Steps", fontsize=11)
    plt.ylabel("Cross-Entropy Loss", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("assets/loss_curve.png", dpi=150)
    plt.show()

# 2. Analyze Layer-wise Sparsity
test_phrase = "To be, or not to be, that is the question:\nWhether 'tis nobler in the mind to suffer"
test_bytes = list(test_phrase.encode("utf-8"))
input_tensor = torch.tensor([test_bytes], dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    _, _, activations = model(input_tensor, return_activations=True)

layer_sparsities = []
for layer_idx, act in enumerate(activations):
    sparsity_pct = (act == 0.0).float().mean().item() * 100.0
    active_neurons_avg = (act > 0.0).sum(dim=-1).float().mean().item()
    layer_sparsities.append(sparsity_pct)
    print(f"Layer {layer_idx + 1:02d} | Sparsity: {sparsity_pct:5.2f}% | Active Neurons: {active_neurons_avg:5.1f} / 1536")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(layer_sparsities) + 1), layer_sparsities, color="#2ca02c", edgecolor="black", alpha=0.85)
axes[0].set_ylim(0, 100)
axes[0].set_xlabel("Layer Index")
axes[0].set_ylabel("Neuron Sparsity (% Silent)")
axes[0].set_title("Biological Sparsity Across BDH Layers", fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

sample_len = min(30, input_tensor.shape[1])
heat_data = activations[0][0, :sample_len, :50].cpu().numpy().T
im = axes[1].imshow(heat_data, aspect="auto", cmap="viridis")
fig.colorbar(im, ax=axes[1], label="Neuron Firing Rate")
axes[1].set_title("Layer 1 Neuron Activations (Top 50 Neurons)", fontweight='bold')
axes[1].set_xlabel("Character / Byte Position")
axes[1].set_ylabel("Neuron Index")
char_labels = [chr(b) if 32 <= b <= 126 else '·' for b in test_bytes[:sample_len]]
axes[1].set_xticks(range(sample_len))
axes[1].set_xticklabels(char_labels, rotation=90)
plt.tight_layout()
plt.savefig("assets/bdh_activation_analysis.png", dpi=150)
plt.show()

## Step 7: Standard Causal Transformer Baseline Comparison

In [ ]:
from hatchling.models.transformer import StandardTransformer

tf_model = StandardTransformer(vocab_size=256, d_model=384, n_heads=6, d_ff=1536, n_layers=6, max_seq_len=128).to(device)
tf_params = sum(p.numel() for p in tf_model.parameters())
print(f"Standard Transformer: {tf_params:,} parameters (~{tf_params / 1e6:.2f}M)")

## Extension 1: Synaptic Plasticity vs Frozen Transformer Weights

In [ ]:
from hatchling.models.plasticity import BDHPlasticBlock
from hatchling.models.transformer import TransformerStaticBlock

sample_phrase = "To be, or not to be, that is the question: Whether 'tis nobler in the mind"
tokens = [ord(c) for c in sample_phrase]
emb = torch.nn.Embedding(256, 64)
bdh_layer = BDHPlasticBlock(d_model=64, n_neurons=128)
tf_layer = TransformerStaticBlock(d_model=64)

initial_bdh = bdh_layer.base_synapse.clone()
initial_tf = tf_layer.q_proj.weight.clone()
bdh_layer.reset_synapses()

bdh_drifts = []
tf_drifts = []
for tok in tokens:
    tok_vec = emb(torch.tensor([tok])).unsqueeze(0)
    _ = bdh_layer.forward_step(tok_vec)
    _ = tf_layer.forward_step(tok_vec)
    bdh_drifts.append(torch.norm(bdh_layer.dynamic_synapse - initial_bdh).item())
    tf_drifts.append(torch.norm(tf_layer.q_proj.weight - initial_tf).item())

print(f"Summary after {len(tokens)} tokens:")
print(f" • Transformer Weight Drift: {tf_drifts[-1]:.6f} (Strictly 0.0 - Static)")
print(f" • BDH Synaptic State Drift: {bdh_drifts[-1]:.4f} (Connections strengthened dynamically)")

## Extension 2: Neuromorphic Energy & Sparsity Profiling (SynOps vs FLOPs)

In [ ]:
from hatchling.analysis.energy import compute_energy_metrics, print_energy_table, plot_energy_comparison

metrics = compute_energy_metrics(empirical_density=0.1406)
print_energy_table(metrics)
plot_energy_comparison(metrics, save_path="assets/energy_profiling.png")

## Extension 3: Stabilized Homeostatic Plasticity (Oja's Normalization Rule)

In [ ]:
from hatchling.models.plasticity import BDHOjaBlock

sample_phrase = "To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer the slings and arrows"
tokens = [ord(c) for c in sample_phrase]
emb = torch.nn.Embedding(256, 64)
oja_layer = BDHOjaBlock(d_model=64, n_neurons=128, eta=0.005)
uncon_layer = BDHPlasticBlock(d_model=64, n_neurons=128, eta=0.05, decay=0.98)

initial_oja = oja_layer.base_synapse.clone()
initial_uncon = uncon_layer.base_synapse.clone()
oja_drifts, uncon_drifts = [], []

for tok in tokens:
    tok_vec = emb(torch.tensor([[tok]]))
    _ = oja_layer.forward_step(tok_vec)
    _ = uncon_layer.forward_step(tok_vec)
    oja_drifts.append(torch.norm(oja_layer.dynamic_synapse - initial_oja).item())
    uncon_drifts.append(torch.norm(uncon_layer.dynamic_synapse - initial_uncon).item())

print(f"Max Drift under Naive Hebbian: {max(uncon_drifts):.2e} (Runaway Feedback)")
print(f"Final Bounded Drift under Oja: {oja_drifts[-1]:.4f} (Bounded, Stable Dynamics)")

## Extension 4: Multi-Step Graph Reachability Benchmark

In [ ]:
from hatchling.analysis.graph_benchmark import generate_graph_sample

sample_text, sample_label = generate_graph_sample(chain_len=2)
print("Sample Graph Query:")
print(sample_text)
print(f"Target Label: {sample_label}")

## Extension 5: Monosemantic Neuron Interpretability Dashboard

In [ ]:
from hatchling.analysis.interpretability import plot_interpretability_dashboard

# Visualize specialist neurons (punctuation, capitalization, word boundary)
plot_interpretability_dashboard(
    model=model,
    punct_neuron=1509,
    punct_ratio=153.26,
    upper_neuron=250,
    upper_ratio=58.65,
    space_neuron=1260,
    space_ratio=36.85,
    save_path="assets/monosemantic_interpretability_dashboard.png",
    device=device
)